# Finding Molecules Similar in Behavior/Function to Rapamycin - Data Extraction and Exploration

In this notebook, I want to go through the data and do some EDA to try to understand how the data works, and what we can do about it

Here are some initial notes from looking online: 

Questions I have are:

- What are morgan fingerprints?

Looking online, there are kernels that extract features of a molecule, hash them, and use the hash to determine bits that should be set. Typical fingerprint sizes are 1-4K bits.

To calculate similarity between fingerprints, we have the Tanimoto similarity function: 

$Tani(V_i , V_j) = \frac{V_j • V_j}{\sum_b V_{ib} + \sum_b V_{jb} - V_j • V_j}$

This function is one of the different kinds of algorithms used to determine similarity

- How do we load in the morgan fingerprints of each data molecule?

We can potentially do this using RDKit
- How do we manipulate the morgan fingerprints of each molecule?

In [ ]:
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import PandasTools, AllChem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from pathlib import Path 


We gotta start off being able to load in the file/s

In [2]:
# Get the project root string to build off of
def find_project_root():
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Project root (with .git) not found")

In [3]:
# Get the name of the sdf file you're looking for from the data file (in case you don't want to actually load everything)
def get_sdf_file_name(name: str):
    PROJECT_ROOT = find_project_root()
    DATA_DIR = PROJECT_ROOT / "data" 
    return DATA_DIR / name

In [4]:
# Load in the sdf file into a pandas dataframe
def load_sdf_file(name: str):
    file_str = get_sdf_file_name(name)
    df = PandasTools.LoadSDF(file_str)
    return df

In [8]:
# Load in a test sdf file to see if we can even load a small sdf file
df = load_sdf_file('test.sdf')

In [ ]:
df.head()
# Looking here, the 'ROMol' column has 'rdkit.Chem.rdchem.Mol' objects
# This can be used to get the Morgan Fingerprints

,PUBCHEM_COMPOUND_CID,PUBCHEM_COMPOUND_CANONICALIZED,PUBCHEM_CACTVS_COMPLEXITY,PUBCHEM_CACTVS_HBOND_ACCEPTOR,PUBCHEM_CACTVS_HBOND_DONOR,PUBCHEM_CACTVS_ROTATABLE_BOND,PUBCHEM_CACTVS_SUBSKEYS,PUBCHEM_IUPAC_OPENEYE_NAME,PUBCHEM_IUPAC_CAS_NAME,PUBCHEM_IUPAC_NAME_MARKUP,...,PUBCHEM_BOND_UDEF_STEREO_COUNT,PUBCHEM_ISOTOPIC_ATOM_COUNT,PUBCHEM_COMPONENT_COUNT,PUBCHEM_CACTVS_TAUTO_COUNT,PUBCHEM_COORDINATE_TYPE,PUBCHEM_BONDANNOTATIONS,ID,ROMol,PUBCHEM_XLOGP3_AA,PUBCHEM_NONSTANDARDBOND
0,135,1,125,3,2,1,AAADcYBgMAAAAAAAAAAAAAAAAAAAAAAAAAAwAAAAAAAAAA...,4-hydroxybenzoic acid,4-hydroxybenzoic acid,4-hydroxybenzoic acid,...,0,0,1,-1,1\n5\n255,4 5 8\n4 6 8\n5 8 8\n6 9 8\n7 8 8\n7...,135,<rdkit.Chem.rdchem.Mol object at 0xfebc91323450>,NaN,NaN
1,338,1,133,3,2,1,AAADcYBgMAAAAAAAAAAAAAAAAAAAAAAAAAAwAAAAAAAAAA...,2-hydroxybenzoic acid,2-hydroxybenzoic acid,2-hydroxybenzoic acid,...,0,0,1,-1,1\n5\n255,4 5 8\n4 6 8\n5 7 8\n6 8 8\n7 9 8\n8...,338,<rdkit.Chem.rdchem.Mol object at 0xfebc9136c890>,NaN,NaN
2,1983,1,139,2,2,1,AAADccByMAAAAAAAAAAAAAAAAAAAAAAAAAAwAAAAAAAAAA...,N-(4-hydroxyphenyl)acetamide,N-(4-hydroxyphenyl)acetamide,<I>N</I>-(4-hydroxyphenyl)acetamide,...,0,0,1,-1,1\n5\n255,4 5 8\n4 6 8\n5 7 8\n6 8 8\n7 9 8\n8...,1983,<rdkit.Chem.rdchem.Mol object at 0xfebc9136cc10>,NaN,NaN
3,2244,1,212,4,1,3,AAADccBwOAAAAAAAAAAAAAAAAAAAAAAAAAAwAAAAAAAAAA...,2-acetoxybenzoic acid,2-acetyloxybenzoic acid,2-acetyloxybenzoic acid,...,0,0,1,-1,1\n5\n255,5 6 8\n5 7 8\n6 8 8\n7 9 8\n8 10 8\n...,2244,<rdkit.Chem.rdchem.Mol object at 0xfebc9136cb30>,NaN,NaN
4,4064,1,212,4,2,8,AAADceBzOAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,[2-(carbamoyloxymethyl)-2-methyl-pentyl] carba...,carbamic acid [2-(carbamoyloxymethyl)-2-methyl...,[2-(carbamoyloxymethyl)-2-methylpentyl] carbamate,...,0,0,1,-1,1\n5\n255,NaN,4064,<rdkit.Chem.rdchem.Mol object at 0xfebc9136cac0>,NaN,NaN


In [ ]:
Mols = df[['PUBCHEM_IUPAC_OPENEYE_NAME', 'ROMol']].copy()
Mols.head()

,PUBCHEM_IUPAC_OPENEYE_NAME,ROMol
0,4-hydroxybenzoic acid,<rdkit.Chem.rdchem.Mol object at 0xfebc91323450>
1,2-hydroxybenzoic acid,<rdkit.Chem.rdchem.Mol object at 0xfebc9136c890>
2,N-(4-hydroxyphenyl)acetamide,<rdkit.Chem.rdchem.Mol object at 0xfebc9136cc10>
3,2-acetoxybenzoic acid,<rdkit.Chem.rdchem.Mol object at 0xfebc9136cb30>
4,[2-(carbamoyloxymethyl)-2-methyl-pentyl] carba...,<rdkit.Chem.rdchem.Mol object at 0xfebc9136cac0>


This is an example of how to use Tanimoto and Cosine Similarity

In [51]:
# Created molecules with smiles for CO2 and CCO 
mol1 = Chem.MolFromSmiles('COO') 
mol2 = Chem.MolFromSmiles('CCO')

# Create a generator for morgan fingerprint
fpgen = GetMorganGenerator(radius=2, fpSize=2048)

# Create the fingerprints
fp1 = fpgen.GetFingerprint(mol1)
fp2 = fpgen.GetFingerprint(mol2)

# Get the similarities
tsimilarity = DataStructs.TanimotoSimilarity(fp1, fp2) # Above 0.85 is similar
csimilarity = DataStructs.CosineSimilarity(fp1, fp2) # -1 to 1 where 1 is the same direction
print(f"Tanimoto Similarity: {tsimilarity}")
print(f"Cosine similarity: {csimilarity}")



Tanimoto Similarity: 0.2
Cosine similarity: 0.3333333333333333
